# CT-FM reproducible workflow



1. Installation
2. Reference downloads
3. GWAS summary statistic download and organization
4. S-LDSC runs
5. CT-FM runs
6. CT-FM-SNP runs

Some steps assume `mamba`/`conda` and the repository layout from the CT-FM project.

## 1. Installation

Run these cells from a shell-enabled notebook environment.

In [ ]:
%%bash
set -euo pipefail

git clone https://github.com/ArtemKimUSC/CTFM.git
cd CTFM
mamba env create -f install/ctfm.yml

## 2. Test the conda environment

These checks verify that LDSC and SuSiE are available.

In [ ]:
%%bash
set -euo pipefail

source "$(conda info --base)/etc/profile.d/conda.sh"
conda activate ctfm

python ~/bin/ldsc/ldsc.py -h
Rscript -e "library(susieR)"
conda deactivate

## 3. Download reference files

In [ ]:
%%bash
set -euo pipefail

cd CTFM
bash install/download_reference.sh
bash install/download_ctfmsnp_reference.sh

## 4. Download GWAS summary statistics and organize the 63 traits

In [ ]:
%%bash
set -euo pipefail

cd CTFM
mkdir -p sumstats_ready
cd sumstats_ready

wget -O sumstats_indep107.tgz "https://zenodo.org/records/10515792/files/sumstats_indep107.tgz?download=1"
tar -xzf sumstats_indep107.tgz

while read -r line; do
  mv "sumstats_107/${line}.sumstats.gz" .
done < traits_63.txt

## 5. Run S-LDSC

In [ ]:
%%bash
set -euo pipefail

cd CTFM
source "$(conda info --base)/etc/profile.d/conda.sh"
conda activate ctfm

while read -r line; do
  bash scripts/2_launch_SLDSC_default.sh "$line"
done < sumstats_ready/traits_63.txt

conda deactivate

## 6. Run CT-FM

In [ ]:
%%bash
set -euo pipefail

cd CTFM
dir="$(pwd)"
source "$(conda info --base)/etc/profile.d/conda.sh"
conda activate ctfm

while read -r line; do
  Rscript scripts/3_launch_susie_default.R "$dir" "$line"
done < sumstats_ready/traits_63.txt

conda deactivate

## 7. Run CT-FM-SNP

### 7A. Get overlapping annotations for SNPs

In [ ]:
%%bash
set -euo pipefail

cd CTFM
source "$(conda info --base)/etc/profile.d/conda.sh"
conda activate ctfm

while read -r line; do
  mkdir -p out/ctfmsnp/"$line"
  bash scripts/4_CTFMSNP_default_annots.sh     "data/UKBB/${line}_finemapped_snps_out.bed"     "out/ctfmsnp/${line}"
done < data/UKBB/list_UKBB_rs.txt

conda deactivate

### 7B. Run CT-FM-SNP

In [ ]:
%%bash
set -euo pipefail

cd CTFM
dir="$(pwd)"
source "$(conda info --base)/etc/profile.d/conda.sh"
conda activate ctfm

while read -r line; do
  cd "out/ctfmsnp/${line}"
  for rs in *out; do
    Rscript "${dir}/scripts/3_launch_susie_default.R" "$dir" "$line"
  done
  cd "$dir"
done < data/UKBB/list_UKBB_rs.txt

conda deactivate

## Notes

- This notebook preserves the original bash workflow.
- If you plan to run it in Google Colab, you may need to adapt the conda installation step, since Colab does not provide `mamba`/`conda` by default.
- The CT-FM-SNP loop above mirrors your original command structure; if the downstream script expects a specific SNP identifier or output filename, you may want to parameterize that explicitly before sharing the notebook.